In [11]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages.utils import trim_messages, count_tokens_approximately

In [12]:
load_dotenv()

True

In [13]:
llm = ChatGroq(model = 'openai/gpt-oss-safeguard-20b')

In [14]:
MAX_TOKENS = 150

In [15]:
def call_model(state: MessagesState):
    
    messages = trim_messages(
        state['messages'],
        strategy = 'last',
        token_counter = count_tokens_approximately,
        max_tokens = MAX_TOKENS
    )
    
    print('Current token count -> ',count_tokens_approximately(messages = messages))
    print('\n\nContext sent to LLM -> ',"\n".join(msg.content for msg in messages))
    
    response = llm.invoke(messages)
    
    return {'messages':[response]}

In [16]:
builder = StateGraph(MessagesState)
builder.add_node('call_model',call_model)
builder.add_edge(START,'call_model')

In [17]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer = checkpointer)

In [18]:
config = {'configurable':{'thread_id':'chat-1'}}

result = graph.invoke(
    {'messages':[{'role':'user','content':'Hi, my name is Saurabh'}]},
    config,
)

result['messages'][-1].content

Current token count ->  10


Context sent to LLM ->  Hi, my name is Saurabh


'Hello Saurabh! 👋 How can I help you today?'

In [19]:
result = graph.invoke(
    {'messages':[{'role':'user','content':'I am learning langgraph.'}]},
    config,
)

result['messages'][-1].content

Current token count ->  36


Context sent to LLM ->  Hi, my name is Saurabh
Hello Saurabh! 👋 How can I help you today?
I am learning langgraph.


'That’s awesome! LangGraph is a powerful framework for building conversational AI workflows. How can I help you get started? Are you looking for an overview, a quick tutorial, help with a specific example, or something else? Let me know what you’d like to dive into!'

In [20]:
result = graph.invoke(
    {'messages':[{'role':'user','content':'Can you explain short term memory to me.'}]},
    config,
)

result['messages'][-1].content

Current token count ->  122


Context sent to LLM ->  Hi, my name is Saurabh
Hello Saurabh! 👋 How can I help you today?
I am learning langgraph.
That’s awesome! LangGraph is a powerful framework for building conversational AI workflows. How can I help you get started? Are you looking for an overview, a quick tutorial, help with a specific example, or something else? Let me know what you’d like to dive into!
Can you explain short term memory to me.


'### Short‑Term Memory (STM) – A Quick Overview\n\n| Aspect | What it means | Why it matters for LangGraph |\n|--------|----------------|------------------------------|\n| **Definition** | The part of memory that holds information for a brief period (seconds to a few minutes). | In a conversation, STM is what keeps track of the most recent utterances, user intent, and context that the LLM needs to answer correctly. |\n| **Capacity** | Roughly 7\u202f±\u202f2 “chunks” of information (Miller’s law), but LLMs can process far more tokens. | LangGraph can store a limited number of recent messages or state variables in a “short‑term buffer” before they’re archived or discarded. |\n| **Duration** | A few seconds to a few minutes without rehearsal. | When a user says “I’d like to book a flight” and then “from New York to Boston,” the second request is still in STM. |\n| **Functions** | • **Working with current context** (e.g., pronoun resolution) <br>• **Immediate decision making** (e.g., next

In [21]:
result = graph.invoke(
    {'messages':[{'role':'user','content':'What is my name?'}]},
    config,
)

result['messages'][-1].content

Current token count ->  8


Context sent to LLM ->  What is my name?


'I don’t have that information—could you remind me?'

In [22]:
result = graph.invoke(
    {'messages':[{'role':'user','content':'My name is Saurabh.'}]},
    config,
)

result['messages'][-1].content

Current token count ->  35


Context sent to LLM ->  What is my name?
I don’t have that information—could you remind me?
My name is Saurabh.


'Nice to meet you, Saurabh! How can I help you today?'